# Model Building

## **1. Import libraries**

In [1]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold
from lazypredict.Supervised import LazyRegressor

## **2. Load the data set**


In [7]:
df = pd.read_csv('..\Data\CSVs\Dopamine_D2_receptor_04_bioactivity_data_model.csv')

<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:1: SyntaxWarning: invalid escape sequence '\D'
C:\Users\HP\AppData\Local\Temp\ipykernel_3972\2342767154.py:1: SyntaxWarning: invalid escape sequence '\D'
  df = pd.read_csv('..\Data\CSVs\Dopamine_D2_receptor_04_bioactivity_data_model.csv')


In [8]:
X = df.drop('pIC50', axis=1)
Y = df.pIC50

In [9]:
X.shape

(1570, 1029)

In [10]:
Y.shape

(1570,)

Remove low variance features

In [13]:

selection = VarianceThreshold(threshold=0.05)
X_filtered = selection.fit_transform(X)

print(f"Original number of features: {X.shape[1]}")
print(f"Number of features after low variance extraction: {X_filtered.shape[1]}")


Original number of features: 1029
Number of features after low variance extraction: 216


split the data

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X_filtered, Y, test_size=0.2, random_state=42)

### Train the lazyRegrossor models

In [15]:
%%time
reg = LazyRegressor(ignore_warnings=True, custom_metric=None)

models, predictions = reg.fit(X_train, X_test, y_train, y_test)

CPU times: total: 30.5 s
Wall time: 21.4 s


In [16]:
print(models)

                               Adjusted R-Squared   R-Squared       RMSE  \
Model                                                                      
HistGradientBoostingRegressor           -0.000247    0.690019   0.891942   
GradientBoostingRegressor               -0.025280    0.682261   0.903035   
SVR                                     -0.269383    0.606613   1.004800   
NuSVR                                   -0.284348    0.601975   1.010705   
RandomForestRegressor                   -0.374682    0.573980   1.045645   
PoissonRegressor                        -0.392896    0.568336   1.052549   
KNeighborsRegressor                     -0.396824    0.567119   1.054032   
BayesianRidge                           -0.413468    0.561960   1.060294   
BaggingRegressor                        -0.426110    0.558043   1.065024   
AdaBoostRegressor                       -0.438129    0.554318   1.069503   
LassoLarsIC                             -0.474962    0.542903   1.083113   
ElasticNetCV

### Increase the variance threshold

In [18]:

selection = VarianceThreshold(threshold=0.1)
X_filtered2 = selection.fit_transform(X)

print(f"Original number of features: {X.shape[1]}")
print(f"Number of features after low variance extraction: {X_filtered2.shape[1]}")


Original number of features: 1029
Number of features after low variance extraction: 87


In [19]:
X_train2, X_test2, y_train2, y_test2 = train_test_split(X_filtered2, Y, test_size=0.2, random_state=42)

In [20]:
%%time
reg = LazyRegressor(ignore_warnings=True, custom_metric=None)

models, predictions = reg.fit(X_train2, X_test2, y_train2, y_test2)

CPU times: total: 13.6 s
Wall time: 9.49 s


In [21]:
print(models)

                               Adjusted R-Squared   R-Squared       RMSE  \
Model                                                                      
HistGradientBoostingRegressor            0.601094    0.711972   0.859779   
GradientBoostingRegressor                0.600473    0.711524   0.860447   
RandomForestRegressor                    0.486700    0.629374   0.975298   
AdaBoostRegressor                        0.448716    0.601948   1.010739   
SVR                                      0.412648    0.575905   1.043280   
NuSVR                                    0.407888    0.572469   1.047498   
BaggingRegressor                         0.401150    0.567603   1.053442   
DecisionTreeRegressor                    0.392221    0.561156   1.061266   
KNeighborsRegressor                      0.370015    0.545123   1.080479   
MLPRegressor                             0.292549    0.489189   1.144984   
BayesianRidge                            0.264799    0.469152   1.167225   
LassoLarsIC 

### We get a very good results R-Squared(71%,71% annd 62%) we will select the top 3 models

Adjusted R-Squared is now positive (~0.60). This tells us that you successfully removed the "curse of dimensionality" (the useless features). our model is now mathematically sound and is no longer being penalized for having hundreds of near-zero variance bit columns.

In the context of drug discovery and predicting pIC50 from molecular fingerprints, an $R^2$ above 0.70 on an unseen test set is generally considered a highly successful baseline model.

It means your model is successfully capturing about 71% of the patterns that determine how a molecule binds to the Dopamine D2 receptor.